# ==============================================================================
# KAGGLE RUNNER: INDOTOXIC2024 DUAL-INPUT CNN EXPERIMENTS
# ==============================================================================
Notebook runner otomatis untuk melatih model klasifikasi teks toksisitas IndoToxic2024
menggunakan akselerasi GPU Kaggle (NVIDIA T4 / P100).

### Alur Kerja:
1. **Autentikasi & Clone**: Clone repository IndoToxic (`Vox`) dari branch aktif via Kaggle Secrets.
2. **Dependensi & Lingkungan**: Instalasi dependensi `requirements.txt` dan setup GPU memory growth.
3. **Data Curation & Preprocessing**: Kurasi 13.995 baris (3.995 toxic + 10.000 non-toxic stratified topic).
4. **Training Standalone**: Latih Context-Aware Dual-Input Multi-Kernel CNN 30 Epochs (Tri-Stage LR Scheduler).
5. **Evaluasi & Error Pouching**: Hitung Macro-F1, Recall, AUC-ROC, Confusion Matrix, dan simpan FP/FN ke ZIP.


In [ ]:
# ==============================================================================
# CELL 1 — Clone Repository IndoToxic (Vox) via Kaggle Secrets
# ==============================================================================
import os
import subprocess
from kaggle_secrets import UserSecretsClient

KAGGLE_WORK_DIR = '/kaggle/working'
REPO_NAME = 'Vox'
BRANCH_NAME = 'joshua'  # Ganti dengan branch target jika diperlukan
PROJECT_DIR = os.path.join(KAGGLE_WORK_DIR, REPO_NAME)

os.chdir(KAGGLE_WORK_DIR)

# Ambil GitHub Token dari Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{github_token}@github.com/Josshua-DSA/{REPO_NAME}.git" if github_token else f"https://github.com/Josshua-DSA/{REPO_NAME}.git"
    print(f"🔑 Token GitHub: {'Terdeteksi' if github_token else 'Kosong (Menggunakan Public Clone)'}")
except Exception as e:
    repo_url = f"https://github.com/Josshua-DSA/{REPO_NAME}.git"
    print(f"⚠️ Secrets tidak tersedia, mencoba public clone: {e}")

if not os.path.exists(PROJECT_DIR):
    print(f"🚀 Melakukan cloning repository {REPO_NAME} branch {BRANCH_NAME}...")
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH_NAME, repo_url],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"❌ Gagal clone: {result.stderr}")
    else:
        print("✅ Clone repository berhasil!")
else:
    print("ℹ️ Folder repository sudah ada di /kaggle/working, memperbarui commit...")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"📂 Direktori aktif saat ini: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 2 — Instalasi Dependensi & Verifikasi GPU Kaggle
# ==============================================================================
import sys
import os

# 1. Install dependencies inti proyek
!pip install -q ruff scikit-learn seaborn matplotlib tqdm python-dotenv

# 2. Konfigurasi Lingkungan GPU TensorFlow & Redam Log Probe cuDNN
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(f"Memory growth warning: {e}")
    print(f"⚡ Akselerasi GPU Aktif: {gpus}")
else:
    print("⚠️ GPU tidak terdeteksi! Pastikan Accelerator di settings Kaggle diaktifkan (GPU T4 x 2 atau P100).")


In [ ]:
# ==============================================================================
# CELL 3 — Setup Path & Impor Modul Proyek IndoToxic
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils.config import Config
from src.utils.seed import set_seed
from src.preprocessing.agreement import AnnotatorAgreementAggregator
from src.preprocessing.normalizer import SlangNormalizer
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder
from src.preprocessing.topic_encoder import TopicEncoder
from src.evaluation.metrics import MetricCalculator
from src.evaluation.error_analysis import ErrorAnalyzer

cfg = Config()
set_seed(cfg.SEED)

print("=== KONFIGURASI PARAMETER INDOTOXIC ===")
print(f"Project Root       : {ROOT_DIR}")
print(f"Random Seed        : {cfg.SEED}")
print(f"Vocab Size         : {cfg.VOCAB_SIZE:,} kata")
print(f"Max Sequence Len   : {cfg.MAX_LEN} token")
print(f"Text Embedding     : FastText ({cfg.EMBEDDING_DIM}d)")
print(f"CNN Kernels/Filters: k={cfg.FILTER_SIZES}, filters={cfg.NUM_FILTERS}")
print(f"Topic Embedding    : {cfg.NUM_TOPICS} kategori -> {cfg.TOPIC_EMBEDDING_DIM}d")
print(f"Loss Function      : Binary Focal Loss (gamma={cfg.FOCAL_GAMMA}, alpha={cfg.FOCAL_ALPHA})")
print("Training Schedule  : 30 Epochs (5 Warmup + 20 Main + 5 Cooldown)")


In [ ]:
# ==============================================================================
# CELL 4 — Kurasi Data (13.995 Baris) & Stratified Splitting (70:15:15)
# ==============================================================================
from sklearn.model_selection import train_test_split

# 1. Muat Raw Data (28.448 baris)
raw_path = ROOT_DIR / cfg.RAW_DATA_CSV
df_raw = pd.read_csv(raw_path)
print(f"[1/3] Raw data dimuat: {len(df_raw):,} baris")

# 2. Agregasi Majority Voting Anotator
aggregator = AnnotatorAgreementAggregator(threshold=0.5)
df_consensus = aggregator.process_dataframe(df_raw, text_col="text", vote_col="toxicity_label")
df_consensus["topic"] = df_raw["topic"].fillna("UNKNOWN").astype(str)

# 3. Kurasi 13.995 baris: 3.995 Toxic (100%) + 10.000 Non-Toxic (Stratified Topic)
toxic_full = df_consensus[df_consensus["label"] == 1].copy()
nontoxic_pool = df_consensus[df_consensus["label"] == 0].copy()

topic_counts = nontoxic_pool["topic"].value_counts()
rare_topics = topic_counts[topic_counts <= 1].index
rare_df = nontoxic_pool[nontoxic_pool["topic"].isin(rare_topics)]
common_df = nontoxic_pool[~nontoxic_pool["topic"].isin(rare_topics)]

n_target_common = 10000 - len(rare_df)
sampled_common, _ = train_test_split(
    common_df,
    train_size=n_target_common,
    stratify=common_df["topic"],
    random_state=cfg.SEED
)
nontoxic_sampled = pd.concat([sampled_common, rare_df]).sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)
df_curated = pd.concat([toxic_full, nontoxic_sampled]).sample(frac=1.0, random_state=cfg.SEED).reset_index(drop=True)
print(f"[2/3] Kurasi data selesai: {len(df_curated):,} baris (Toxic: {len(toxic_full):,}, Non-toxic: {len(nontoxic_sampled):,})")

# 4. Stratified Split 70% Train, 15% Val, 15% Test
train_df, temp_df = train_test_split(
    df_curated,
    test_size=0.30,
    stratify=df_curated["label"],
    random_state=cfg.SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=cfg.SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"[3/3] Partisi Split: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")


In [ ]:
# ==============================================================================
# CELL 5 — Preprocessing Teks & Tokenisasi Anti-Leakage
# ==============================================================================
normalizer = SlangNormalizer(config=cfg)
cleaner = TextCleaner()

def preprocess_pipeline(texts: list[str]) -> list[str]:
    results = []
    for t in texts:
        cleaned = cleaner.clean(t)
        normalized = normalizer.normalize(cleaned)
        results.append(normalized)
    return results

print("🧹 Menjalankan preprocessing sanitasi teks...")
train_df["text_preprocessed"] = preprocess_pipeline(train_df["text"].tolist())
val_df["text_preprocessed"] = preprocess_pipeline(val_df["text"].tolist())
test_df["text_preprocessed"] = preprocess_pipeline(test_df["text"].tolist())

# Tokenizer fit HANYA pada data Train
tokenizer = TextTokenizer(vocab_size=cfg.VOCAB_SIZE, oov_token=cfg.OOV_TOKEN, pad_token=cfg.PAD_TOKEN)
tokenizer.fit(train_df["text_preprocessed"].tolist())

padder = SequencePadder(max_len=cfg.MAX_LEN, padding="post", truncating="post")
X_text_train = padder.pad(tokenizer.texts_to_sequences(train_df["text_preprocessed"].tolist()))
X_text_val = padder.pad(tokenizer.texts_to_sequences(val_df["text_preprocessed"].tolist()))
X_text_test = padder.pad(tokenizer.texts_to_sequences(test_df["text_preprocessed"].tolist()))

topic_encoder = TopicEncoder(canonical_topics=cfg.CANONICAL_TOPICS)
X_topic_train = topic_encoder.transform(train_df["topic"]).reshape(-1, 1)
X_topic_val = topic_encoder.transform(val_df["topic"]).reshape(-1, 1)
X_topic_test = topic_encoder.transform(test_df["topic"]).reshape(-1, 1)

y_train = train_df["label"].to_numpy(dtype=np.int32)
y_val = val_df["label"].to_numpy(dtype=np.int32)
y_test = test_df["label"].to_numpy(dtype=np.int32)
print("✅ Ekstraksi fitur dan tensor selesai!")


In [ ]:
# ==============================================================================
# CELL 6 — Model Architecture & Training Standalone Eksplisit (30 Epochs)
# ==============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

# 1. Setup Parameter
VOCAB_SIZE = int(cfg.VOCAB_SIZE)
MAX_LEN = int(cfg.MAX_LEN)
EMBEDDING_DIM = int(cfg.EMBEDDING_DIM)
FILTER_SIZES = [3, 4, 5]
NUM_FILTERS = 128
NUM_TOPICS = len(cfg.CANONICAL_TOPICS)
TOPIC_EMBEDDING_DIM = 32
DENSE_UNITS = 128
DROPOUT_FUSION = 0.5
DROPOUT_DENSE = 0.3
BATCH_SIZE = 32
TOTAL_EPOCHS = 30
WARMUP_EPOCHS = 5
MAIN_EPOCHS = 20
BASE_LR = float(cfg.LEARNING_RATE)

# 2. Graf Functional API Multi-Kernel CNN
text_input = layers.Input(shape=(MAX_LEN,), dtype="int32", name="text_input")
text_emb = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, name="trainable_embedding")(text_input)

conv_blocks = []
for k_size in FILTER_SIZES:
    conv = layers.Conv1D(filters=NUM_FILTERS, kernel_size=k_size, activation="relu", padding="valid", name=f"conv1d_k{k_size}")(text_emb)
    pool = layers.GlobalMaxPooling1D(name=f"gmp_k{k_size}")(conv)
    conv_blocks.append(pool)
text_vec = layers.Concatenate(name="text_vec")(conv_blocks)  # 384d

topic_input = layers.Input(shape=(1,), dtype="int32", name="topic_input")
topic_emb = layers.Embedding(input_dim=NUM_TOPICS, output_dim=TOPIC_EMBEDDING_DIM, name="topic_embedding")(topic_input)
topic_vec = layers.Flatten(name="topic_vec")(topic_emb)      # 32d

fusion = layers.Concatenate(name="fusion_vec")([text_vec, topic_vec])  # 416d
x = layers.Dropout(DROPOUT_FUSION, name="dropout_fusion")(fusion)
x = layers.Dense(DENSE_UNITS, activation="relu", name="dense_hidden")(x)
x = layers.Dropout(DROPOUT_DENSE, name="dropout_dense")(x)
output_prob = layers.Dense(1, activation="sigmoid", name="output_prob")(x)

model = models.Model(inputs=[text_input, topic_input], outputs=output_prob, name="Context_Aware_Dual_Input_CNN")

# 3. Binary Focal Loss
def binary_focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    y_true = tf.cast(y_true, tf.float32)
    p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
    alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
    focal_weight = alpha_t * tf.pow(1.0 - p_t, gamma)
    return tf.reduce_mean(focal_weight * (-tf.math.log(p_t)))

model.compile(
    optimizer=optimizers.Adam(learning_rate=BASE_LR),
    loss=binary_focal_loss,
    metrics=["accuracy"],
    jit_compile=False
)

# 4. Tri-Stage Scheduler
def tri_stage_lr_schedule(epoch, lr):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * float(epoch + 1) / float(WARMUP_EPOCHS)
    elif epoch < (WARMUP_EPOCHS + MAIN_EPOCHS):
        return BASE_LR
    else:
        step = epoch - (WARMUP_EPOCHS + MAIN_EPOCHS) + 1
        return BASE_LR * (0.5 ** step)

lr_callback = tf.keras.callbacks.LearningRateScheduler(tri_stage_lr_schedule, verbose=1)

# 5. Training Standalone
print(f"🚀 Memulai Training {TOTAL_EPOCHS} Epochs pada GPU Kaggle...")
raw_history = model.fit(
    [X_text_train, X_topic_train],
    y_train,
    validation_data=([X_text_val, X_topic_val], y_val),
    batch_size=BATCH_SIZE,
    epochs=TOTAL_EPOCHS,
    callbacks=[lr_callback],
    verbose=1
)
print("🎉 Training 30 Epochs Selesai 100%!")


In [ ]:
# ==============================================================================
# CELL 7 — Evaluasi Test Set & Ekspor Error Pouching (FP & FN)
# ==============================================================================
import os
import shutil

# 1. Inferensi Test Set
y_prob = model.predict([X_text_test, X_topic_test], batch_size=32, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

calc = MetricCalculator()
metrics = calc.compute_all(y_true=y_test, y_pred=y_pred, y_prob=y_prob)

print("================ HASIL EVALUASI MODEL TEST SET (KAGGLE GPU) ================")
print(f"Macro-F1 Score    : {metrics['macro_f1']:.4f}  (Target riset >= 0.78)")
print(f"Macro-Precision   : {metrics['precision_macro']:.4f}")
print(f"Recall (Toxic)    : {metrics['per_class']['toxic']['recall']:.4f}  (Target riset >= 0.75)")
print(f"AUC-ROC           : {metrics['auc_roc']:.4f}")
print(f"Accuracy          : {metrics['accuracy']:.4f}")
print(f"Per-Class Toxic F1: {metrics['per_class']['toxic']['f1']:.4f}")
print(f"Confusion Matrix  :\n{metrics['confusion_matrix']}")
print("============================================================================")

# 2. Pouching FP & FN
test_df_eval = test_df.copy()
test_df_eval["pred_label"] = y_pred
test_df_eval["pred_prob"] = y_prob

df_fp = test_df_eval[(test_df_eval["label"] == 0) & (test_df_eval["pred_label"] == 1)].copy()
df_fn = test_df_eval[(test_df_eval["label"] == 1) & (test_df_eval["pred_label"] == 0)].copy()
df_fp = df_fp.sort_values(by="pred_prob", ascending=False)
df_fn = df_fn.sort_values(by="pred_prob", ascending=True)

output_dir = Path("/kaggle/working/outputs/error_analysis")
output_dir.mkdir(parents=True, exist_ok=True)

cols = ["topic", "label", "pred_label", "pred_prob", "text", "text_preprocessed"]
df_fp[[c for c in cols if c in df_fp.columns]].to_csv(output_dir / "false_positives.csv", index=False)
df_fn[[c for c in cols if c in df_fn.columns]].to_csv(output_dir / "false_negatives.csv", index=False)

# Simpan bobot model
model_dir = Path("/kaggle/working/outputs/models")
model_dir.mkdir(parents=True, exist_ok=True)
model.save(str(model_dir / "cnn_model_kaggle.keras"))

# Packing hasil penting ke ZIP untuk didownload
shutil.make_archive("/kaggle/working/indotoxic_results", "zip", "/kaggle/working/outputs")
print("\n📦 Hasil evaluasi, artefak model, dan error analysis berhasil dipak ke: /kaggle/working/indotoxic_results.zip")
